# Apache Avro - Rust

All 6 Rust examples from [docs/avro.md](https://platob.github.io/yggdryl/avro/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{MediaType, MimeType, Value, avro, json};

let schema = json::from_str(
    r#"{"type": "record", "name": "trade", "fields": [
        {"name": "symbol", "type": "string"},
        {"name": "quantity", "type": "long"},
        {"name": "price", "type": ["null", "double"], "default": null}
    ]}"#,
)?;
let rows = [
    json::from_str(r#"{"symbol": "AAPL", "quantity": 100, "price": 187.5}"#)?,
    json::from_str(r#"{"symbol": "MSFT", "quantity": 25, "price": null}"#)?,
];

let mut handle = Buffer::new();
handle.set_media_type(MediaType::new(MimeType::AVRO));
avro::write_container(&mut handle, &schema, &[("source", "docs")], &rows)?;

let container = avro::read_container(&handle)?;
assert_eq!(container.get("source"), Some("docs"));
assert_eq!(container.schema.kind(), "record");
assert_eq!(container.rows.len(), 2);
assert_eq!(
    container.rows[0].get_key_str("symbol").and_then(Value::as_str),
    Some("AAPL")
);
assert!(container.rows[1].get_key_str("price").is_some_and(Value::is_null));

## Schemas, canonical form, and fingerprints

In [ ]:
use yggdryl::avro::Schema;

let schema = Schema::from_str(
    r#"{"type": "record", "name": "trade", "doc": "one fill", "fields": [
        {"name": "symbol", "type": "string"},
        {"name": "qty", "type": "long", "field-id": 2}
    ]}"#,
)?;

// The canonical form strips docs, defaults, and unknown attributes, and is
// what every implementation fingerprints.
assert_eq!(
    schema.to_canonical_form(),
    r#"{"name":"trade","type":"record","fields":[{"name":"symbol","type":"string"},{"name":"qty","type":"long"}]}"#
);

// The 64-bit Rabin fingerprint names the schema in caches and in the
// single-object framing; this value matches the reference implementations.
assert_eq!(schema.fingerprint().to_le_bytes()[0], 0xF5);

// The JSON the schema was parsed from round-trips verbatim, so the
// unmodeled `field-id` survives.
let text = String::from_utf8(yggdryl::json::to_vec(&schema.to_json())?)?;
assert!(text.contains("field-id"));

## Logical types decode as what they mean

In [ ]:
use yggdryl::enums::TimeUnit;
use yggdryl::io::Buffer;
use yggdryl::{Timezone, Value, avro, json};

let schema = json::from_str(
    r#"{"type": "record", "name": "row", "fields": [
        {"name": "day", "type": {"type": "int", "logicalType": "date"}},
        {"name": "at", "type": {"type": "long", "logicalType": "timestamp-micros"}},
        {"name": "price", "type": {"type": "bytes", "logicalType": "decimal",
                                    "precision": 10, "scale": 2}}
    ]}"#,
)?;
let row = Value::from_mapping([
    (Value::from("day"), Value::Date(19_782)),
    (
        Value::from("at"),
        Value::Timestamp(1_700_000_000_000_000, TimeUnit::Microsecond, Timezone::UTC),
    ),
    (Value::from("price"), Value::Decimal(18_750, 2)),
])?;

let mut handle = Buffer::new();
avro::write_container(&mut handle, &schema, &[], &[row.clone()])?;
assert_eq!(avro::read_container(&handle)?.rows[0], row);

## Reading with a different schema

In [ ]:
use yggdryl::avro::Schema;
use yggdryl::io::Buffer;
use yggdryl::{Value, avro, json};

// The writer recorded three fields; the reader wants two - one renamed, one
// promoted, plus a field the writer never knew, filled from its default.
let writer = json::from_str(
    r#"{"type": "record", "name": "trade", "fields": [
        {"name": "symbol", "type": "string"},
        {"name": "qty", "type": "int"},
        {"name": "venue", "type": "string"}
    ]}"#,
)?;
let reader = Schema::from_str(
    r#"{"type": "record", "name": "trade", "fields": [
        {"name": "quantity", "aliases": ["qty"], "type": "long"},
        {"name": "note", "type": "string", "default": "none"}
    ]}"#,
)?;

let mut handle = Buffer::new();
avro::write_container(
    &mut handle,
    &writer,
    &[],
    &[json::from_str(r#"{"symbol": "AAPL", "qty": 100, "venue": "XNAS"}"#)?],
)?;

let container = avro::read_container_resolved(&handle, &reader)?;
assert_eq!(
    container.rows[0].get_key_str("quantity").and_then(Value::as_i64),
    Some(100),
    "matched through the alias, promoted int to long"
);
assert_eq!(
    container.rows[0].get_key_str("note").and_then(Value::as_str),
    Some("none")
);
assert_eq!(container.rows[0].len(), 2, "unwanted writer fields are skipped");

## Streaming a large container

In [ ]:
use yggdryl::io::Buffer;
use yggdryl::{Value, avro, json};

let schema = json::from_str(r#"{"type": "record", "name": "row", "fields": [
    {"name": "id", "type": "long"}]}"#)?;
let rows: Vec<Value> = (0..3)
    .map(|id| Value::from_mapping([(Value::from("id"), Value::from(id))]))
    .collect::<Result<_, _>>()?;

let mut handle = Buffer::new();
avro::write_container(&mut handle, &schema, &[], &rows)?;

let mut blocks = avro::read_blocks(&handle)?;
assert_eq!(blocks.schema().kind(), "record");
while let Some(block) = blocks.next_block()? {
    // A block is handed back still compressed; decoding is the caller's
    // choice, so skipping a block costs nothing.
    assert_eq!(block.rows()?.len() as u64, block.count());
}

## Single-object encoding

In [ ]:
use yggdryl::avro::Schema;
use yggdryl::{Value, avro};

let schema = Schema::from_str(r#"{"type": "record", "name": "tick", "fields": [
    {"name": "price", "type": "double"}]}"#)?;
let value = Value::from_mapping([(Value::from("price"), Value::from(187.5))])?;

let framed = avro::to_single_object_vec(&schema, &value)?;
assert_eq!(&framed[..2], &[0xC3, 0x01], "the single-object marker");
assert_eq!(avro::from_single_object_slice(&framed, &schema)?, value);

// A frame from a different schema is refused naming both fingerprints.
let other = Schema::from_str("\"long\"")?;
let message = avro::from_single_object_slice(&framed, &other)
    .unwrap_err()
    .to_string();
assert!(message.contains("fingerprint"));